<a href="https://colab.research.google.com/github/athitthiyan/Learning_Gen_AI/blob/main/Colab_1_LangChain_Agent_STM_LTM_AZURE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Colab 1 — LangChain Agent with Short-Term & Long-Term Memory + Tools

**Workshop: Agentic AI — Full-Day Training**

---

## Learning Objectives

By the end of this lab you will be able to:
1. Understand the difference between **Short-Term Memory (STM)** and **Long-Term Memory (LTM)** in LangChain agents
2. Implement `ConversationBufferMemory` (STM) and a `ChromaDB vector store` (LTM)
3. Wire up real tools: **web search**, **Python REPL**, and a **custom calculator**
4. Build and run a **ReAct agent** that reasons, acts, and remembers
5. *(Extension)* Swap STM for `ConversationSummaryMemory`, add SQL retrieval, self-critique loop, and streaming UI

---

## Architecture Overview

```
User Prompt
     │
     ▼
┌─────────────────────────────────────────────┐
│               AgentExecutor                  │
│                                             │
│  ┌─────────────┐    ┌────────────────────┐  │
│  │  ReAct LLM  │◄──►│  STM (Buffer/      │  │
│  │  (Claude /  │    │  Summary Memory)   │  │
│  │  GPT-4o)    │    └────────────────────┘  │
│  └──────┬──────┘                            │
│         │ tool calls                        │
│  ┌──────▼──────────────────────────────┐    │
│  │            Tool Router              │    │
│  │  [Search]  [PythonREPL]  [Calc]     │    │
│  └─────────────────────────────────────┘    │
└─────────────────────────────────────────────┘
     │
     ▼
┌─────────────────────┐
│  LTM: Chroma VectorDB│  ← stores every Q&A pair
│  (persistent across  │    retrieved at query time
│   sessions)         │
└─────────────────────┘
```

**STM** = the rolling conversation window this session  
**LTM** = semantic vector store that persists across sessions

---

## ⏱ Timing
| Section | Time |
|---------|------|
| Setup & Install | 10 min |
| Part 1 — STM Agent | 20 min |
| Part 2 — LTM + STM Agent | 20 min |
| Part 3 — Full Agent with Tools | 15 min |
| Extension Tasks | 30+ min |


---
## 📦 Section 0 — Installation & Setup

In [1]:
# Install all required packages (LangChain v1 + langchain-classic stack)
# Run this cell first — it takes ~2 minutes. On first run, RESTART THE RUNTIME afterwards.
!pip install -q -U \
    langchain \
    langchain-classic \
    langchain-openai \
    langchain-community \
    langchain-experimental \
    langchain-huggingface \
    langchain-chroma \
    langchain-tavily \
    chromadb \
    tavily-python \
    sentence-transformers \
    tiktoken \
    faiss-cpu

print("✅ Packages installed — on first run, RESTART THE RUNTIME, then run from the top.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18

In [2]:
# ⚠️ This cell is intentionally disabled.
# It originally pinned `langchain==0.3.25`, which DOWNGRADES LangChain and conflicts
# with the v1 + langchain-classic imports used throughout this notebook
# (e.g. `from langchain_classic.agents import AgentExecutor`).
# Everything you need is installed in the cell above — just skip this one.
print("⏭️  Skipped (legacy version pin removed to avoid conflicts).")

⏭️  Skipped (legacy version pin removed to avoid conflicts).


In [3]:
# import os

# ─── API Keys ────────────────────────────────────────────────────────────────
# Set your keys here OR use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
# os.environ["TAVILY_API_KEY"]    = userdata.get("TAVILY_API_KEY")

# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # ← replace
# os.environ["TAVILY_API_KEY"]    = "tvly-..."      # ← replace

# ─── Model ───────────────────────────────────────────────────────────────────
# from langchain_anthropic import ChatAnthropic

# llm = ChatAnthropic(
#     model="claude-sonnet-4-6",
#     temperature=0,
#     max_tokens=2048,
# )

import os
from google.colab import userdata
from langchain_openai import AzureChatOpenAI

# Pull credentials from Colab Secrets (🔑 in the left sidebar)
AZURE_ENDPOINT    = userdata.get('AZURE_OPENAI_ENDPOINT')    # https://ey-learn-azure-openai.openai.azure.com/
AZURE_API_KEY     = userdata.get('AZURE_OPENAI_KEY')
AZURE_DEPLOYMENT  = userdata.get('AZURE_OPENAI_DEPLOYMENT')  # your GPT-4o *deployment* name
AZURE_API_VERSION = '2024-12-01-preview'

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    azure_deployment=AZURE_DEPLOYMENT,
    openai_api_version=AZURE_API_VERSION,
    openai_api_key=AZURE_API_KEY,
    temperature=0,        # deterministic for financial queries
    max_tokens=2048,
    timeout=30,
)
print('✅ Azure GPT-4o LLM ready')


✅ Azure GPT-4o LLM ready


---
## 🧠 Part 1 — Short-Term Memory (STM)

**Short-Term Memory** lives in the LLM's context window.  
It keeps the current conversation history so the agent can refer back to earlier messages.

### Two STM strategies we'll try:
| Class | How it works | Best for |
|-------|-------------|----------|
| `ConversationBufferMemory` | Keeps **all** messages verbatim | Short sessions, debugging |
| `ConversationSummaryMemory` | Summarises older exchanges to save tokens | Long sessions, production |

We'll start with `ConversationBufferMemory`.


In [4]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

# ── Build STM ────────────────────────────────────────────────────────────────
stm = ConversationBufferMemory(
    memory_key="history",   # key injected into the prompt template
    return_messages=True,   # return as list[BaseMessage] not a string
)

# Quick test with a simple ConversationChain (no tools yet)
conv_chain = ConversationChain(llm=llm, memory=stm, verbose=True)

print("=== Turn 1 ===")
r1 = conv_chain.predict(input="Hi! My name is Alex and I'm an ML engineer.")
print(r1)

print("\n=== Turn 2 ===")
r2 = conv_chain.predict(input="What did I just tell you about myself?")
print(r2)


/tmp/ipykernel_9565/3286736783.py:5: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  stm = ConversationBufferMemory(
/tmp/ipykernel_9565/3286736783.py:11: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conv_chain = ConversationChain(llm=llm, memory=stm, verbose=True)


=== Turn 1 ===


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: Hi! My name is Alex and I'm an ML engineer.
AI:

> Finished chain.
Hi Alex! It's great to meet you. I'm an AI designed to assist with all sorts of questions and conversations. Being an ML engineer sounds fascinating—you're probably working on some cutting-edge projects! Are you focused on a specific area, like computer vision, natural language processing, or reinforcement learning? Or maybe something else entirely? I'd love to hear more about what you do!

=== Turn 2 ===


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots o

In [5]:
# ── Inspect what's stored in STM ─────────────────────────────────────────────
print("Messages in STM buffer:")
for msg in stm.chat_memory.messages:
    role = msg.__class__.__name__
    print(f"  [{role}] {msg.content[:120]}")


Messages in STM buffer:
  [HumanMessage] Hi! My name is Alex and I'm an ML engineer.
  [AIMessage] Hi Alex! It's great to meet you. I'm an AI designed to assist with all sorts of questions and conversations. Being an ML
  [HumanMessage] What did I just tell you about myself?
  [AIMessage] You told me that your name is Alex and that you're an ML (Machine Learning) engineer.


### 🔍 Observation
Notice how the agent correctly recalls "Alex" and "ML engineer" — those facts lived in the STM buffer.

**Limitation**: if the conversation runs long, the context window fills up and older facts are silently dropped.  
This is exactly why we need **Long-Term Memory**.


---
## 📚 Part 2 — Long-Term Memory (LTM) with ChromaDB

**Long-Term Memory** stores information in a **vector database** that persists across sessions.  
At each turn the agent:
1. Embeds the user's query
2. Retrieves the top-k most relevant past Q&A pairs
3. Injects them into the prompt as additional context

### Why vector search?
Semantic similarity — not keyword matching. A query about "revenue last quarter" will retrieve  
"Q3 sales figures" even if those exact words weren't used.


In [6]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.memory import VectorStoreRetrieverMemory

# ── Embeddings ────────────────────────────────────────────────────────────────
# Default: local sentence-transformers model (fast, 384-dim, no API key, no cost).
# The LTM works out of the box with this — no Azure embeddings deployment required.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# ── (Optional) Azure OpenAI embeddings instead ───────────────────────────────
# If you have a SEPARATE embeddings deployment in Azure (e.g. text-embedding-3-small),
# comment out the HuggingFaceEmbeddings block above and use this instead:
#
# from langchain_openai import AzureOpenAIEmbeddings
# embeddings = AzureOpenAIEmbeddings(
#     azure_endpoint=AZURE_ENDPOINT,
#     azure_deployment=userdata.get("AZURE_OPENAI_EMBED_DEPLOYMENT"),  # embeddings deployment name
#     openai_api_version=AZURE_API_VERSION,
#     openai_api_key=AZURE_API_KEY,
# )

# ── Persistent Chroma vector store ───────────────────────────────────────────
# persist_directory keeps the DB between Colab sessions if you mount Drive
vectorstore = Chroma(
    collection_name="agent_ltm",
    embedding_function=embeddings,
    persist_directory="./chroma_ltm",   # remove for in-memory only
)

# ── Wrap as a retriever ───────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

ltm = VectorStoreRetrieverMemory(
    retriever=retriever,
    memory_key="ltm_context",
    return_docs=False,   # return as formatted string
)

print(f"✅ LTM ready — collection: '{vectorstore._collection.name}'")
print(f"   Docs currently stored: {vectorstore._collection.count()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ LTM ready — collection: 'agent_ltm'
   Docs currently stored: 0


/tmp/ipykernel_9565/801584309.py:36: LangChainDeprecationWarning: The class `VectorStoreRetrieverMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  ltm = VectorStoreRetrieverMemory(


In [7]:
# ── Manually seed some long-term memories ────────────────────────────────────
# In production these would be saved automatically after each agent run.

seed_memories = [
    {"input": "What is the user's name?",         "output": "Alex"},
    {"input": "What does the user work on?",       "output": "Machine Learning engineering at a fintech startup"},
    {"input": "What stack does the user prefer?",  "output": "Python, PyTorch, LangChain, Postgres"},
    {"input": "What project is the user working on?",
     "output": "Building a RAG pipeline over internal financial documents"},
    {"input": "What LLM provider does the user prefer?",
     "output": "Anthropic Claude for reasoning tasks, OpenAI for embeddings"},
]

for mem in seed_memories:
    ltm.save_context({"input": mem["input"]}, {"output": mem["output"]})

print(f"✅ Seeded {len(seed_memories)} memories")
print(f"   Total docs in LTM: {vectorstore._collection.count()}")


✅ Seeded 5 memories
   Total docs in LTM: 5


In [8]:
# ── Test LTM retrieval ────────────────────────────────────────────────────────
query = "What framework does Alex use for building AI systems?"
retrieved = ltm.load_memory_variables({"prompt": query})

print(f"Query:  {query}\n")
print("Retrieved LTM context:")
print(retrieved["ltm_context"])


Query:  What framework does Alex use for building AI systems?

Retrieved LTM context:
input: What does the user work on?
output: Machine Learning engineering at a fintech startup
input: What is the user's name?
output: Alex
input: What LLM provider does the user prefer?
output: Anthropic Claude for reasoning tasks, OpenAI for embeddings


### ✅ What just happened?
1. We embedded the query: *"What framework does Alex use..."*
2. Chroma found the 3 most semantically similar stored Q&A pairs
3. They were returned as context to inject into the next prompt

**Try changing the query** — notice the retrieval changes based on meaning, not exact keywords.


---
## 🔧 Part 3 — Defining Tools

We'll give the agent three tools:

| Tool | What it does | When the agent uses it |
|------|-------------|------------------------|
| `TavilySearch` | Real-time web search | Current events, facts it doesn't know |
| `PythonREPL` | Execute arbitrary Python code | Calculations, data manipulation, plotting |
| `Calculator` | Safe arithmetic evaluation | Simple math without running full Python |


In [9]:
pip install langchain-community langchain-experimental tavily-python

In [10]:
pip install langchain-tavily

In [11]:
from langchain_tavily import TavilySearch
from langchain_experimental.tools import PythonREPLTool
from langchain_core.tools import Tool
import math, ast, operator
from langchain_experimental.utilities import PythonREPL
import os, getpass
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key: ")

# ── Tool 1: Web Search ────────────────────────────────────────────────────────
search_tool = TavilySearch(max_results=4)
search_tool.description = (
    "Search the web for real-time information. "
    "Use for current events, recent data, or anything beyond your training cutoff."
)

# ── Tool 2: Python REPL ───────────────────────────────────────────────────────
_repl = PythonREPL()
python_tool = Tool(
    name="PythonREPL",
    func=_repl.run,
    description="Execute Python code in a sandboxed REPL. "
    "Use for calculations, data analysis, string manipulation, or any computation. "
    "Input must be valid Python. Print your results.",
)

# ── Tool 3: Safe Calculator ───────────────────────────────────────────────────
def safe_calc(expression: str) -> str:
    """Evaluate a simple arithmetic expression safely (no exec/eval tricks)."""
    allowed_ops = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg,
    }
    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.n
        elif isinstance(node, ast.BinOp):
            return allowed_ops[type(node.op)](_eval(node.left), _eval(node.right))
        elif isinstance(node, ast.UnaryOp):
            return allowed_ops[type(node.op)](_eval(node.operand))
        else:
            raise ValueError(f"Unsupported expression: {ast.dump(node)}")
    try:
        tree = ast.parse(expression, mode="eval")
        result = _eval(tree.body)
        return str(round(result, 6))
    except Exception as e:
        return f"Error: {e}"

calculator_tool = Tool(
    name="Calculator",
    func=safe_calc,
    description=(
        "Evaluate arithmetic expressions: +, -, *, /, **. "
        "Input: a plain math expression like '(1200 * 1.15) / 12'. "
        "Use this for simple arithmetic; use PythonREPL for complex logic."
    ),
)

tools = [search_tool, python_tool, calculator_tool]
print(f"✅ {len(tools)} tools ready: {[t.name for t in tools]}")


/tmp/ipykernel_9565/4279601849.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool


Tavily API key: ··········
✅ 3 tools ready: ['tavily_search', 'PythonREPL', 'Calculator']


---
## 🤖 Part 4 — Assembling the Full Agent (STM + LTM + Tools)

We now combine everything:
- **STM** (`ConversationBufferMemory`) — rolling conversation window
- **LTM** (`VectorStoreRetrieverMemory`) — semantic recall from past sessions
- **Tools** — search, code execution, calculator

We use a **custom prompt** that injects both memory types, then a `ReAct` agent loop.


In [12]:
%pip install -qU langchain-classic

In [13]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# ── Custom ReAct prompt that uses BOTH memory types ───────────────────────────
REACT_TEMPLATE = """You are a helpful, knowledgeable AI research assistant with access to tools.

### Long-Term Memory (from past sessions)
{ltm_context}

### Current Conversation (Short-Term Memory)
{history}

### Available Tools
{tools}

### Tool Names
{tool_names}

### Instructions
- Reason step-by-step using the format below
- Use tools when you need real-time data or computation
- Reference Long-Term Memory when relevant to personalise your response
- Be concise but thorough

### Format (STRICT — always follow this)
Question: the input question
Thought: what you need to do
Action: the tool name (must be one of [{tool_names}])
Action Input: the input to the tool
Observation: the tool result
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: your complete response

Begin!

Question: {input}
Thought: {agent_scratchpad}
"""

prompt = PromptTemplate(
    input_variables=["input", "history", "ltm_context", "tools", "tool_names", "agent_scratchpad"],
    template=REACT_TEMPLATE,
)

print("✅ Custom ReAct prompt created")
print(f"   Input variables: {prompt.input_variables}")


✅ Custom ReAct prompt created
   Input variables: ['agent_scratchpad', 'history', 'input', 'ltm_context', 'tool_names', 'tools']


In [14]:
%pip install -qU langchain-openai

In [15]:
# from langchain.chat_models import init_chat_model

import os
from google.colab import userdata
from langchain_openai import AzureChatOpenAI

# Pull credentials from Colab Secrets (🔑 in the left sidebar)
AZURE_ENDPOINT    = userdata.get('AZURE_OPENAI_ENDPOINT')
AZURE_API_KEY     = userdata.get('AZURE_OPENAI_KEY')
AZURE_DEPLOYMENT  = userdata.get('AZURE_OPENAI_DEPLOYMENT')   # deployment name, not model name
AZURE_API_VERSION = '2024-12-01-preview'

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    azure_deployment=AZURE_DEPLOYMENT,
    openai_api_version=AZURE_API_VERSION,
    openai_api_key=AZURE_API_KEY,
    temperature=0,        # deterministic for financial queries
    max_tokens=2048,
    timeout=30,
)
print('✅ Azure GPT-4o LLM ready')

# ── STM for this session ──────────────────────────────────────────────────────
session_stm = ConversationBufferMemory(
    memory_key="history",
    return_messages=False,   # string format for ReAct template
    input_key="input",
    output_key="output",
)

# ── Build the ReAct agent ─────────────────────────────────────────────────────
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# ── AgentExecutor wires it all together ───────────────────────────────────────
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=session_stm,           # STM is managed automatically
    max_iterations=8,             # safety: cap the ReAct loop
    handle_parsing_errors=True,   # recover gracefully from format errors
    verbose=True,                 # show the full Thought→Action→Observation trace
    return_intermediate_steps=True,
)

print("✅ AgentExecutor ready")
print(f"   Max iterations: {executor.max_iterations}")


✅ Azure GPT-4o LLM ready
✅ AgentExecutor ready
   Max iterations: 8


In [16]:
# ── Helper: inject LTM context before each run ───────────────────────────────
def run_agent(user_input: str, show_steps: bool = False) -> str:
    """
    Run the agent with both STM (auto) and LTM (injected from Chroma).

    Args:
        user_input: the user's question
        show_steps: if True, print intermediate tool steps
    Returns:
        the agent's final answer
    """
    # 1. Retrieve relevant LTM context for this query
    ltm_vars = ltm.load_memory_variables({"prompt": user_input})
    ltm_context = ltm_vars.get("ltm_context", "No relevant past context found.")

    # 2. Run the agent
    result = executor.invoke({
        "input": user_input,
        "ltm_context": ltm_context,
    })

    # 3. Save this exchange to LTM for future sessions
    ltm.save_context(
        {"input": user_input},
        {"output": result["output"]},
    )

    # 4. Optional: show intermediate steps
    if show_steps and "intermediate_steps" in result:
        print("\n📋 Tool calls made:")
        for action, observation in result["intermediate_steps"]:
            print(f"  🔧 {action.tool}({action.tool_input!r})")
            print(f"     → {str(observation)[:200]}")

    return result["output"]

print("✅ run_agent() helper ready")


✅ run_agent() helper ready


---
## 🧪 Part 5 — Test the Agent

Run the cells below one at a time. Observe:
- How the **LTM context** is injected at the top of each run
- The **Thought → Action → Observation** loop in verbose output
- How **STM** makes follow-up questions work naturally


In [17]:
# ── Test 1: Personalised response using LTM ───────────────────────────────────
print("=" * 60)
print("TEST 1: Does the agent remember who it's talking to?")
print("=" * 60)

answer = run_agent(
    "What AI framework should I use for my project?",
    show_steps=True
)
print("\n🤖 Final Answer:")
print(answer)


TEST 1: Does the agent remember who it's talking to?


> Entering new AgentExecutor chain...
Thought: The user works on Machine Learning engineering at a fintech startup and prefers Python, PyTorch, LangChain, and Postgres. I need to understand the specific requirements of the project to recommend an AI framework. I'll ask for more details.

Final Answer: Could you provide more details about your project? For example, is it focused on NLP, computer vision, time-series analysis, or something else? Additionally, are there specific constraints or goals, such as scalability, real-time inference, or integration with existing systems?

> Finished chain.

📋 Tool calls made:

🤖 Final Answer:
Could you provide more details about your project? For example, is it focused on NLP, computer vision, time-series analysis, or something else? Additionally, are there specific constraints or goals, such as scalability, real-time inference, or integration with existing systems?


In [18]:
# ── Test 2: Multi-step with web search + calculation ─────────────────────────
print("=" * 60)
print("TEST 2: Web search + arithmetic in one task")
print("=" * 60)

answer = run_agent(
    "What is the current population of India? "
    "Calculate what 0.5% of that would be, and convert to millions.",
    show_steps=True
)
print("\n🤖 Final Answer:")
print(answer)


TEST 2: Web search + arithmetic in one task


> Entering new AgentExecutor chain...
Thought: I need to find the current population of India using real-time data. Then, I will calculate 0.5% of that population and convert the result into millions.

Action: tavily_search  
Action Input: "current population of India 2023"  
{'query': 'current population of India 2023', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.worldometers.info/world-population/india-population', 'title': 'India Population (2026) - Worldometer', 'content': 'The current population of India is 1.476.152.019 as of tiistaina 16. India population is equivalent to 17.79% of the total world population. 2023 1,438,069,596', 'score': 0.9441409, 'raw_content': None}, {'url': 'https://www.macrotrends.net/global-metrics/countries/ind/india/population', 'title': 'India Population (1950-2026) - Macrotrends', 'content': 'Total population for India in 2024. Total population for India in 2

It seems the Calculator tool is encountering a formatting issue. I will switch to PythonREPL for the calculation.

Action: PythonREPL  
Action Input: "population = 1428627663; percentage = 0.5; result = (percentage / 100) * population; result_in_millions = result / 1_000_000; result_in_millions"  
SyntaxError('unterminated string literal (detected at line 1)', ('<string>', 1, 145, 'population = 1428627663; percentage = 0.5; result = (percentage / 100) * population; result_in_millions = result / 1_000_000; result_in_millions"', 1, 145))It seems there was a persistent issue with the input formatting. I will carefully reformat the calculation and retry using PythonREPL.

Action: PythonREPL  
Action Input: "population = 1428627663\npercentage = 0.5\nresult = (percentage / 100) * population\nresult_in_millions = result / 1_000_000\nresult_in_millions"  
SyntaxError('unexpected character after line continuation character', ('<string>', 1, 25, 'population = 1428627663\\npercentage = 0.5\\nres

In [19]:
# ── Test 3: STM in action — follow-up question ───────────────────────────────
print("=" * 60)
print("TEST 3: Follow-up using STM (no re-stating context)")
print("=" * 60)

run_agent("Tell me about the latest developments in transformer architectures.")
answer = run_agent("Which of those would be most relevant to my work?")  # refers to Test 1 LTM + Test 3 STM

print("\n🤖 Final Answer:")
print(answer)


TEST 3: Follow-up using STM (no re-stating context)


> Entering new AgentExecutor chain...
Thought: I need to search for the latest developments in transformer architectures, as this requires real-time information beyond my training cutoff. I will use the `tavily_search` tool to gather recent updates.

Action: tavily_search
Action Input: Latest developments in transformer architectures 2023
{'query': 'Latest developments in transformer architectures 2023', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.nightfall.ai/ai-security-101/transformer-architectures', 'title': 'Transformer Architectures: The Essential Guide - Nightfall AI', 'content': 'Recent developments in Transformer Architectures include the introduction of models such as GPT-3, which has 175 billion parameters and has achieved state-of-', 'score': 0.7668008, 'raw_content': None}, {'url': 'https://jytan.net/blog/2025/transformer-architectures', 'title': 'The Crystallization of Tr

In [20]:
# ── Test 4: Python REPL for data analysis ────────────────────────────────────
print("=" * 60)
print("TEST 4: Agent writes and runs Python code")
print("=" * 60)

answer = run_agent(
    "Generate a list of the first 10 Fibonacci numbers, "
    "compute their sum, and tell me what percentage each number "
    "contributes to the total.",
    show_steps=True
)
print("\n🤖 Final Answer:")
print(answer)


TEST 4: Agent writes and runs Python code


> Entering new AgentExecutor chain...
To solve this, I need to:
1. Generate the first 10 Fibonacci numbers.
2. Compute their sum.
3. Calculate the percentage contribution of each number to the total.

Action: PythonREPL
Action Input: 
```python
# Generate the first 10 Fibonacci numbers
fibonacci = [0, 1]
for _ in range(8):
    fibonacci.append(fibonacci[-1] + fibonacci[-2])

# Compute their sum
total_sum = sum(fibonacci)

# Calculate percentage contribution of each number
percentages = [(num / total_sum) * 100 for num in fibonacci]

fibonacci, total_sum, percentages
```Observation: The Python code execution was not completed due to a missing observation. I will reattempt the computation.

Action: PythonREPL
Action Input: 
```python
# Generate the first 10 Fibonacci numbers
fibonacci = [0, 1]
for _ in range(8):
    fibonacci.append(fibonacci[-1] + fibonacci[-2])

# Compute their sum
total_sum = sum(fibonacci)

# Calculate percentage contributi

In [21]:
# ── Inspect STM after 4 turns ─────────────────────────────────────────────────
print("\n📝 Current STM buffer (last 4 turns):")
history_str = session_stm.load_memory_variables({})["history"]
print(history_str[:2000])

print(f"\n📚 LTM now contains {vectorstore._collection.count()} documents")



📝 Current STM buffer (last 4 turns):
Human: What AI framework should I use for my project?
AI: Could you provide more details about your project? For example, is it focused on NLP, computer vision, time-series analysis, or something else? Additionally, are there specific constraints or goals, such as scalability, real-time inference, or integration with existing systems?
Human: What is the current population of India? Calculate what 0.5% of that would be, and convert to millions.
AI: Agent stopped due to iteration limit or time limit.
Human: Tell me about the latest developments in transformer architectures.
AI: Recent developments in transformer architectures include innovations like FlashAttention, Longformer, Performer, and Mamba for improved memory and speed. Best practices for 2023–2025 include pre-norm layers (RMSNorm), Rotary Positional Embeddings (RoPE), and SwiGLU activations. Techniques like positional interpolation are extending context windows for processing longer sequenc

---
## 🔬 Part 6 — Observe & Compare: STM vs LTM

Run this cell to see the difference between what's in STM vs LTM right now.


In [22]:
from IPython.display import Markdown, display

# ── What's in STM right now? ─────────────────────────────────────────────────
stm_history = session_stm.load_memory_variables({})["history"]

# ── What would LTM retrieve for a given query? ────────────────────────────────
test_query = "What programming languages does Alex use?"
ltm_result = ltm.load_memory_variables({"prompt": test_query})["ltm_context"]

summary_md = f"""
## Memory Comparison

### 🧠 Short-Term Memory (this session)
Contains **{len(session_stm.chat_memory.messages)} messages** from the current conversation.

```
{stm_history[-800:] if len(stm_history) > 800 else stm_history}
```

---

### 📚 Long-Term Memory (Chroma — persists across sessions)
Query: *"{test_query}"*

Retrieved context:
```
{ltm_result}
```

**Total documents in LTM:** {vectorstore._collection.count()}

---

### Key Differences
| Property | STM (Buffer) | LTM (Chroma) |
|----------|-------------|--------------|
| Scope | This session only | Across all sessions |
| Retrieval | Sequential (all messages) | Semantic similarity search |
| Persistence | Lost on session end | Saved to disk |
| Token cost | Grows linearly | Fixed-size injection (top-k) |
| Best for | Context continuity | User facts, past decisions |
"""

display(Markdown(summary_md))



## Memory Comparison

### 🧠 Short-Term Memory (this session)
Contains **10 messages** from the current conversation.

```
er sequences. The field is advancing toward more efficient, scalable, and versatile transformer models.
Human: Which of those would be most relevant to my work?
AI: For your RAG pipeline over internal financial documents, Longformer and Positional Interpolation are most relevant for handling long documents. FlashAttention and Performer could also be beneficial for improving efficiency and scalability.
Human: Generate a list of the first 10 Fibonacci numbers, compute their sum, and tell me what percentage each number contributes to the total.
AI: The first 10 Fibonacci numbers are [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]. Their sum is 89. The percentage contributions are approximately:
- 0: 0%
- 1: 1.12%
- 1: 1.12%
- 2: 2.25%
- 3: 3.37%
- 5: 5.62%
- 8: 8.99%
- 13: 14.61%
- 21: 23.60%
- 34: 38.20%.
```

---

### 📚 Long-Term Memory (Chroma — persists across sessions)
Query: *"What programming languages does Alex use?"*

Retrieved context:
```
input: What is the user's name?
output: Alex
input: What stack does the user prefer?
output: Python, PyTorch, LangChain, Postgres
input: What LLM provider does the user prefer?
output: Anthropic Claude for reasoning tasks, OpenAI for embeddings
```

**Total documents in LTM:** 10

---

### Key Differences
| Property | STM (Buffer) | LTM (Chroma) |
|----------|-------------|--------------|
| Scope | This session only | Across all sessions |
| Retrieval | Sequential (all messages) | Semantic similarity search |
| Persistence | Lost on session end | Saved to disk |
| Token cost | Grows linearly | Fixed-size injection (top-k) |
| Best for | Context continuity | User facts, past decisions |


---
---
# ⚡ Extension Tasks

These tasks are for participants who finish early. Each builds on the core agent above.

---

## ⚡ Extension 1 — Swap to `ConversationSummaryMemory`

`ConversationBufferMemory` keeps *all* messages verbatim.  
For long sessions this wastes tokens. `ConversationSummaryMemory` asks the LLM to  
**summarise older exchanges** and only keeps the summary + last N messages.


In [25]:
# ── Extension 1: ConversationSummaryMemory ────────────────────────────────────
from langchain_classic.memory import ConversationSummaryMemory

summary_stm = ConversationSummaryMemory(
    llm=llm,                    # LLM used to write the summary
    memory_key="history",
    return_messages=False,
    input_key="input",
    output_key="output",
    human_prefix="User",
    ai_prefix="Agent",
)

# Re-build the executor with summary memory
summary_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=summary_stm,
    max_iterations=8,
    handle_parsing_errors=True,
    verbose=False,              # quiet mode so we can focus on memory
    return_intermediate_steps=False,
)

def run_agent_summary(user_input: str) -> str:
    ltm_vars = ltm.load_memory_variables({"prompt": user_input})
    result = summary_executor.invoke({
        "input": user_input,
        "ltm_context": ltm_vars.get("ltm_context", ""),
    })
    return result["output"]

# ── Run several turns and watch the summary grow ──────────────────────────────
for turn in [
    "Tell me about the history of neural networks.",
    "What were the key innovations in the 2010s?",
    "How did attention mechanisms change everything?",
    "What should I read to go deeper on this?",
]:
    run_agent_summary(turn)

# ── Inspect the running summary ───────────────────────────────────────────────
print("📝 Running conversation summary (STM):")
print(summary_stm.buffer or "(no summary yet)")
print(f"\nMessages still in buffer: {len(summary_stm.chat_memory.messages)}")


📝 Running conversation summary (STM):
The user asks about the history of neural networks. The agent explains that neural networks originated in the 1940s with McCulloch and Pitts' artificial neuron model and advanced with Rosenblatt's Perceptron in 1958, though interest waned in the 1970s due to limitations. The 1980s saw renewed interest with backpropagation enabling multi-layer networks, followed by CNNs for image processing in the 1990s. The 2000s brought the deep learning revolution, driven by computational power, large datasets, and innovations like ReLU and GPUs, with breakthroughs like AlexNet and RNNs. The 2010s introduced transformative innovations, including the success of AlexNet in image recognition, the transformer architecture revolutionizing NLP with models like BERT and GPT, GANs for generative modeling, and advancements in RNNs and LSTMs for sequential data. Techniques like ReLU activation, batch normalization, and transfer learning improved training efficiency and red

### 💡 Discussion
- How does the summary compare to the raw buffer from Part 5?  
- What information was compressed or lost?  
- When would you choose Summary over Buffer in production?


---
## ⚡ Extension 2 — Add a SQLite Tool (Structured LTM)

Vector search is great for semantic retrieval, but sometimes you need **exact lookups** —  
user IDs, transaction amounts, timestamps. A SQL tool gives the agent structured memory.


In [26]:
# ── Extension 2: SQLite as structured LTM ────────────────────────────────────
import sqlite3, json
from langchain_core.tools import StructuredTool
from pydantic import BaseModel

# ── Create a simple SQLite DB ─────────────────────────────────────────────────
conn = sqlite3.connect(":memory:")   # use a file path for persistence
cur = conn.cursor()

cur.executescript("""
    CREATE TABLE IF NOT EXISTS user_preferences (
        id INTEGER PRIMARY KEY,
        category TEXT,
        key TEXT,
        value TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    INSERT INTO user_preferences (category, key, value) VALUES
        ('tools',    'preferred_llm',       'Claude Sonnet'),
        ('tools',    'preferred_framework', 'LangChain + LangGraph'),
        ('project',  'name',                'FinDoc RAG Pipeline'),
        ('project',  'tech_stack',          'Python, Chroma, FastAPI'),
        ('project',  'deadline',            '2025-09-30');
""")
conn.commit()

# ── Tool: SQL query ───────────────────────────────────────────────────────────
class SQLQueryInput(BaseModel):
    query: str

def run_sql(query: str) -> str:
    """Run a read-only SQL query against the user preferences DB."""
    try:
        # Safety: only allow SELECT
        if not query.strip().upper().startswith("SELECT"):
            return "Error: only SELECT queries are allowed."
        rows = cur.execute(query).fetchall()
        cols = [d[0] for d in cur.description]
        if not rows:
            return "No results found."
        return json.dumps([dict(zip(cols, row)) for row in rows], indent=2)
    except Exception as e:
        return f"SQL Error: {e}"

sql_tool = StructuredTool.from_function(
    func=run_sql,
    name="UserPreferencesDB",
    description=(
        "Query the user's structured preference database using SQL SELECT statements. "
        "Table: user_preferences(id, category, key, value, created_at). "
        "Use this to look up exact user settings, project details, or tool preferences."
    ),
    args_schema=SQLQueryInput,
)

# ── Rebuild with 4 tools ──────────────────────────────────────────────────────
extended_tools = [search_tool, python_tool, calculator_tool, sql_tool]

ext_agent = create_react_agent(llm=llm, tools=extended_tools, prompt=prompt)
ext_executor = AgentExecutor(
    agent=ext_agent,
    tools=extended_tools,
    memory=ConversationBufferMemory(memory_key="history", return_messages=False,
                                    input_key="input", output_key="output"),
    max_iterations=8,
    handle_parsing_errors=True,
    verbose=True,
)

# ── Test it ───────────────────────────────────────────────────────────────────
def run_ext(q):
    ltm_ctx = ltm.load_memory_variables({"prompt": q}).get("ltm_context", "")
    return ext_executor.invoke({"input": q, "ltm_context": ltm_ctx})["output"]

print(run_ext("What framework am I using and when is my project deadline?"))




> Entering new AgentExecutor chain...
Thought: I need to check the user's preferences or project details in the database to determine the framework they are using and their project deadline.

Action: UserPreferencesDB
Action Input: SELECT value FROM user_preferences WHERE category = 'project' AND key = 'framework';
No results found.The database does not have information about the framework you are using. I also need to check if there is any information about your project deadline.

Action: UserPreferencesDB
Action Input: SELECT value FROM user_preferences WHERE category = 'project' AND key = 'deadline';
[
  {
    "value": "2025-09-30"
  }
]The database indicates that your project deadline is September 30, 2025. However, there is no information about the framework you are using. If you'd like, I can help you choose or confirm the framework based on your project's requirements. Let me know!Invalid Format: Missing 'Action:' after 'Thought:'The database indicates that your project deadli

---
## ⚡ Extension 3 — Self-Critique Loop (Reflexion Pattern)

The **Reflexion** pattern asks the agent to evaluate its own answer,  
identify weaknesses, then produce an improved version.


In [27]:
# ── Extension 3: Self-Critique Loop ──────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Critic prompt ─────────────────────────────────────────────────────────────
critic_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a rigorous AI quality reviewer.

Given a question and an agent's answer, provide:
1. A score from 1-10 (10 = perfect)
2. Specific weaknesses (missing facts, logic errors, unclear language)
3. A concrete suggestion for improvement

Format:
SCORE: <number>
WEAKNESSES: <bullet points>
SUGGESTION: <one clear improvement instruction>
"""),
    ("human", "Question: {question}\n\nAnswer: {answer}"),
])

# ── Revision prompt ───────────────────────────────────────────────────────────
revision_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a precise AI assistant. Revise the answer based on the feedback provided."),
    ("human", "Original question: {question}\n\nOriginal answer: {answer}\n\nCritic feedback: {critique}\n\nRevised answer:"),
])

critic_chain = critic_prompt | llm | StrOutputParser()
revision_chain = revision_prompt | llm | StrOutputParser()

def reflexion_run(question: str, max_rounds: int = 2) -> dict:
    """Run the agent, then apply self-critique rounds."""
    print(f"\n{'='*55}")
    print(f"Question: {question}")
    print('='*55)

    # Initial answer from the agent
    ltm_ctx = ltm.load_memory_variables({"prompt": question}).get("ltm_context", "")
    initial = executor.invoke({"input": question, "ltm_context": ltm_ctx})["output"]
    print(f"\n[Round 0 — Initial Answer]\n{initial}")

    current_answer = initial
    history = [{"round": 0, "answer": initial, "score": None, "critique": None}]

    for rnd in range(1, max_rounds + 1):
        # Critique
        critique = critic_chain.invoke({"question": question, "answer": current_answer})
        score_line = [l for l in critique.split("\n") if l.startswith("SCORE:")]
        score = int(score_line[0].split(":")[1].strip()) if score_line else 0
        print(f"\n[Round {rnd} — Critique] Score: {score}/10")
        print(critique)

        if score >= 9:
            print("\n✅ Score threshold reached — stopping early.")
            break

        # Revise
        current_answer = revision_chain.invoke({
            "question": question,
            "answer": current_answer,
            "critique": critique,
        })
        print(f"\n[Round {rnd} — Revised Answer]\n{current_answer}")
        history.append({"round": rnd, "answer": current_answer, "score": score, "critique": critique})

    return {"final_answer": current_answer, "history": history}

# ── Run it ────────────────────────────────────────────────────────────────────
result = reflexion_run(
    "What are the main risks of using LLM-based agents in production financial systems?",
    max_rounds=2
)



Question: What are the main risks of using LLM-based agents in production financial systems?


> Entering new AgentExecutor chain...
Thought: I need to outline the main risks of using LLM-based agents in production financial systems, focusing on areas like accuracy, security, compliance, and scalability. Drawing from my knowledge and the user's fintech context, I will provide a detailed response.

Final Answer: The main risks of using LLM-based agents in production financial systems include:

1. **Accuracy and Reliability**: LLMs can generate plausible but incorrect or nonsensical outputs, which could lead to financial miscalculations or incorrect advice.

2. **Data Privacy and Security**: Financial systems handle sensitive data. Improper handling or leakage of this data by LLMs could lead to compliance violations or breaches.

3. **Regulatory Compliance**: Financial systems must adhere to strict regulations (e.g., GDPR, PCI DSS). LLMs may inadvertently generate outputs that violate t

### 💡 Discussion
- Did the score improve across rounds?
- What kinds of weaknesses did the critic identify?
- Is there a point of diminishing returns? When would you cap the rounds?


---
## ⚡ Extension 4 — Streaming Agent Output to a UI

In production you want to **stream** the agent's intermediate steps to the user  
so they see progress rather than a blank screen for 30 seconds.


In [29]:
# ── Extension 4: Streaming with callbacks ─────────────────────────────────────
from langchain_core.callbacks import BaseCallbackHandler, StdOutCallbackHandler
import time

class StreamingDisplayHandler(BaseCallbackHandler):
    """Custom callback that prints each token/step as it arrives."""

    def on_llm_new_token(self, token: str, **kwargs):
        print(token, end="", flush=True)

    def on_tool_start(self, serialized, input_str, **kwargs):
        tool_name = serialized.get("name", "unknown")
        print(f"\n\n🔧 Calling tool: {tool_name}")
        print(f"   Input: {str(input_str)[:150]}")

    def on_tool_end(self, output, **kwargs):
        print(f"   Result: {str(output)[:200]}")

    def on_agent_action(self, action, **kwargs):
        print(f"\n💭 Thought → {action.log[:300]}")

    def on_agent_finish(self, finish, **kwargs):
        print(f"\n\n✅ Final Answer: {finish.return_values.get('output', '')}")

# ── Build a streaming executor ────────────────────────────────────────────────
streaming_llm = AzureChatOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    azure_deployment=AZURE_DEPLOYMENT,
    openai_api_version=AZURE_API_VERSION,
    openai_api_key=AZURE_API_KEY,
    temperature=0,
    streaming=True,
    callbacks=[StreamingDisplayHandler()],
)

stream_agent = create_react_agent(llm=streaming_llm, tools=tools, prompt=prompt)
stream_executor = AgentExecutor(
    agent=stream_agent,
    tools=tools,
    memory=ConversationBufferMemory(memory_key="history", return_messages=False,
                                    input_key="input", output_key="output"),
    max_iterations=6,
    handle_parsing_errors=True,
    verbose=False,   # using our custom handler instead
)

print("🚀 Running agent with live streaming output...\n")
ltm_ctx = ltm.load_memory_variables({"prompt": "streaming test"}).get("ltm_context", "")
stream_executor.invoke({
    "input": "Search for the latest news about LangChain updates and summarise the top 3 items.",
    "ltm_context": ltm_ctx,
})


🚀 Running agent with live streaming output...

Thought: I need to search for the latest updates about LangChain and summarise the top three relevant news items. I'll use the tavily_search tool for this.

Action: tavily_search
Action Input: Latest LangChain updates October 2023
The search results provide information about recent updates to LangChain, particularly in October 2023. I'll summarise the top three relevant items.

1. **Data Annotation Queues in LangSmith**:
   - LangChain introduced a new feature called "data annotation queues" in LangSmith, their SaaS platform for managing LangChain applications. This feature was launched on October 30, 2023, and aims to streamline the annotation process for data used in LangChain workflows.

2. **Frequent Version Updates**:
   - LangChain has been actively releasing updates, with multiple versions in October 2023. For example, version `0.0.326` was released on October 31, 2023, and version `0.0.325` on October 30, 2023. These updates likely

{'input': 'Search for the latest news about LangChain updates and summarise the top 3 items.',
 'ltm_context': 'input: What LLM provider does the user prefer?\noutput: Anthropic Claude for reasoning tasks, OpenAI for embeddings\ninput: What does the user work on?\noutput: Machine Learning engineering at a fintech startup\ninput: Which of those would be most relevant to my work?\noutput: For your RAG pipeline over internal financial documents, Longformer and Positional Interpolation are most relevant for handling long documents. FlashAttention and Performer could also be beneficial for improving efficiency and scalability.',
 'history': '',
 'output': 'The top three LangChain updates for October 2023 are:\n1. Introduction of "data annotation queues" in LangSmith to improve data management.\n2. Frequent version releases, including `0.0.326` on October 31, 2023.\n3. Regular product updates detailed in their changelog, with a focus on platform enhancements.'}

---
## 🎯 Lab Summary

### What you built

| Component | Class / Tool | Purpose |
|-----------|-------------|---------|
| **STM (Buffer)** | `ConversationBufferMemory` | Full conversation history this session |
| **STM (Summary)** | `ConversationSummaryMemory` | Compressed history — saves tokens |
| **LTM** | `VectorStoreRetrieverMemory` + Chroma | Semantic recall across sessions |
| **Structured LTM** | SQLite + `StructuredTool` | Exact lookup of user facts |
| **Web Search** | `TavilySearchResults` | Real-time grounded answers |
| **Code Execution** | `PythonREPLTool` | Dynamic computation |
| **Calculator** | Custom `Tool` | Safe arithmetic |
| **Self-Critique** | Reflexion chain | Iterative quality improvement |
| **Streaming** | `BaseCallbackHandler` | Live progress display |

---

### Key Takeaways

1. **STM ≠ LTM** — they serve different purposes and should be used together
2. **Buffer vs Summary** — choose based on session length and token budget  
3. **LTM requires a retrieval strategy** — semantic (Chroma) or structured (SQL)  
4. **Reflexion improves quality** — at the cost of latency and tokens  
5. **Streaming is a UX necessity** — always add it before going to production

---

### Next Steps
- 📘 Colab 2: Rebuild this agent natively with the Anthropic SDK (no LangChain)
- 🔁 Compare: latency, cost, reasoning trace quality between the two approaches
- 🚀 Advanced: deploy with LangGraph + `MemorySaver` for production-grade persistence
